In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from churn import config


In [11]:
df=pd.read_parquet(config.CLEAN_DATA_FILE)
X = df.drop(columns=[config.TARGET])
y = df[config.TARGET]

Train-test-split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [13]:
print(f"X_train.shape: {X_train.shape}, X_test.shape: {X_test.shape}")
print(f"churn rate  train={y_train.mean():.4f}  test={y_test.mean():.4f}")

X_train.shape: (5282, 20), X_test.shape: (1761, 20)
churn rate  train=0.2654  test=0.2652


In [14]:
X_train.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
6661,3704-IEAXF,Female,No,Yes,Yes,72,No,No phone service,DSL,No,Yes,No,Yes,Yes,Yes,Two year,No,Credit card (automatic),53.65,3784.00
4811,5175-AOBHI,Female,No,No,No,4,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,46.00,193.60
2193,6922-NCEDI,Male,No,No,Yes,56,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,21.20,1238.65
1904,3143-ILDAL,Male,No,No,No,56,Yes,Yes,Fiber optic,No,Yes,No,Yes,No,Yes,Month-to-month,Yes,Electronic check,94.45,5124.60
6667,0872-NXJYS,Female,No,No,No,9,Yes,No,Fiber optic,No,No,No,No,No,Yes,Month-to-month,Yes,Electronic check,79.55,723.40


Pipeline

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer

from churn.features import collapse_no_internet, build_preprocessor, add_features

In [20]:
def build_pipeline(model, model_type, num_cols=None, cat_cols=None):
    """Raw rows in → churn prediction out. model_type: "lr" or "tree" (picks the preprocessor)."""
    return Pipeline([
        ("collapse", FunctionTransformer(collapse_no_internet)),
        ("features", FunctionTransformer(add_features)),
        ("prep", build_preprocessor(model_type, num_cols, cat_cols)),
        ("model", model),
    ])

In [21]:
pipe_lr = build_pipeline(LogisticRegression(max_iter=1000), model_type="lr").fit(X_train, y_train)

In [24]:
pipe_lr[:-1].transform(X_train.head())

,num__tenure,num__MonthlyCharges,num__num_services,cat__SeniorCitizen_Yes,cat__Partner_Yes,cat__Dependents_Yes,cat__MultipleLines_No,cat__MultipleLines_No phone service,cat__MultipleLines_Yes,cat__InternetService_DSL,...,cat__PaymentMethod_Bank transfer (automatic),cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check,cat__tenure_group_0-6,cat__tenure_group_13-24,cat__tenure_group_25-48,cat__tenure_group_49-72,cat__tenure_group_7-12,cat__has_family_Yes
6661,1.609608,-0.371461,0.855137,0.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4811,-1.151780,-0.625032,-1.071019,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
2193,0.959870,-1.447065,-1.071019,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
1904,0.959870,0.980917,0.213085,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6667,-0.948737,0.487034,-1.071019,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [26]:
from sklearn.model_selection import StratifiedKFold, cross_validate

In [27]:
# Define metrics relevant for imbalanced data
scoring_metrics = {
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

In [30]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_lr = cross_validate(pipe_lr, X_train, y_train, cv=skf, scoring=scoring_metrics, n_jobs = -1)
cv_lr_df = pd.DataFrame(cv_lr)
cv_lr_df

,fit_time,score_time,test_f1,test_precision,test_recall,test_roc_auc,test_pr_auc
0,0.115381,0.052447,0.611670,0.703704,0.540925,0.863088,0.702385
1,0.112803,0.052825,0.653846,0.711297,0.604982,0.867309,0.709270
2,0.111659,0.067373,0.560000,0.636364,0.500000,0.822342,0.591834
3,0.092068,0.050550,0.590164,0.692308,0.514286,0.838483,0.671899
4,0.101371,0.065339,0.549180,0.644231,0.478571,0.839702,0.638713


#### Notes 1: how to read `cv_lr_df` (Logistic Regression, 5-fold CV)

**The table**
- **1 row = 1 fold.** Each fold trains the whole Pipeline on 4/5 of `X_train` and scores the remaining 1/5 (≈ 1,056 customers).
- **`test_*` = the validation fold**, *not* `X_test`. That's sklearn's naming; the real test set stays untouched.
- `fit_time` / `score_time`: seconds to train / to predict + score. Useful later to compare slow vs fast models.

**`.agg(["mean", "std"]).T`**
- **mean** = the score to expect on new customers. Use it to **compare models**.
- **std** = how much the score moves between folds, i.e. how **stable** (trustworthy) the mean is.
- Rule of thumb: if two models' means differ by **less than about one std**, the difference may just be noise from how the folds were split. Don't call a winner on it.

**Is a small std good?** Yes. It means the score doesn't depend on which customers ended up in which fold. For ≈ 1,000 rows per fold:

| std | Reading |
|---|---|
| < 0.02 | Very stable |
| 0.02 – 0.05 | Normal for this data size (our P, R, F1, PR-AUC ≈ 0.03–0.05) |
| > 0.05 | Unstable: check the data or the model before trusting the mean |

**What counts as a good score?**

| Metric | Random model | Our LR | Good on this dataset | Warning sign |
|---|---|---|---|---|
| ROC-AUC | 0.50 | **0.85 ± 0.02** | 0.83 – 0.86 (typical tuned models) | > 0.90 → suspect leakage |
| PR-AUC | 0.265 (= churn rate) | **0.66 ± 0.05** | 0.60 – 0.70 | Near 0.265 → model learned nothing |
| Precision / Recall / F1 | depends on threshold | 0.68 / 0.53 / 0.59 | No fixed range: they change with the threshold (phase 6) | n/a |

ROC-AUC and PR-AUC don't use a cut-off, so they get "good ranges". Precision, recall and F1 here use the default 0.5 cut-off, so they are only a first look.


In [ ]:
cv_lr_df.agg(["mean", "std"]).T   # .T turns it so each metric is a row


,mean,std
fit_time,0.106657,0.009743
score_time,0.057707,0.007975
test_f1,0.592972,0.042036
test_precision,0.677581,0.034811
test_recall,0.527753,0.048755
test_roc_auc,0.846185,0.018720
test_pr_auc,0.662820,0.048543


### Notes 2: what the mean scores mean for the business

**The confusion matrix words** (positive = churn = 1):

| | **Really churns** | **Really stays** |
|---|---|---|
| Model says **churn** (flagged) | **TP**: caught, gets an offer ✅ | **FP**: false alarm → *wasted offer* (cheaper) |
| Model says **stays** | **FN**: missed, leaves → *lost revenue* (expensive) | **TN**: correctly left alone |

**One validation fold in numbers** (≈ 1,056 customers, ≈ 280 real churners), with the mean scores at the 0.5 cut-off:

| Metric | Formula | Mean | In plain words | In one fold |
|---|---|---|---|---|
| **Precision** | TP / (TP + FP) | 0.68 | Of the customers we **flag**, 68% really churn | ≈ 218 flagged → 148 real churners, **70 wasted offers** |
| **Recall** | TP / (TP + FN) | 0.53 | Of all real **churners**, we catch 53% | 148 caught, **132 missed** (lost revenue) |
| **F1** | 2·P·R / (P + R) | 0.59 | One number that balances P and R; low if either one is low | n/a |
| **ROC-AUC** | P(random churner scored higher than random stayer) | 0.85 | The model **ranks** churners above stayers 85% of the time | Uses probabilities, not the cut-off |
| **PR-AUC** | Precision averaged over all recall levels | 0.66 | How good the flagged list is at every list length. Random = 0.265 | ≈ 2.5× better than random |

**Business reading**
- **Recall gets extra weight:** a missed churner (FN) costs more than a wasted offer (FP), but precision keeps the budget and team workload in check. Phase 6 balances the two by cost.
- **Precision protects the budget and the team's time.** It says how many offers are wasted, and the number flagged must fit the retention team's capacity.
- **Precision, recall and F1 change with the threshold.** Lowering the cut-off from 0.5 flags more customers: recall ↑, precision ↓. Phase 6 picks the threshold from the cost of FN vs FP. So, for now, **don't compare models on these three**.
- **ROC-AUC and PR-AUC don't depend on the threshold.** They measure how well the model *orders* customers by risk, which is what the retention team uses ("call the top N"). **Models are ranked on these two in phase 5.**
- **Accuracy is left out on purpose.** Predicting "nobody churns" already scores 73.5% but catches 0 churners.

**Why recall gets more weight than precision: the two mistakes cost different amounts**

| Metric | The mistake it counts | What that mistake costs |
|---|---|---|
| **Recall** = TP / (TP + **FN**) | **FN**: a churner we didn't flag | They leave. We lose their future monthly revenue. |
| **Precision** = TP / (TP + **FP**) | **FP**: a loyal customer we flagged | We give a discount to someone who would have stayed. We lose the cost of one offer. |

Rough dollar figures (illustrative only; phase 1 states the assumptions, phase 6 fixes the real numbers):
- **FN (missed churner):** lost revenue ≈ `MonthlyCharges` × months they'd have stayed × chance the offer would have kept them.
  - Example: $75/month × 12 months × 30% success ≈ **$270**.
- **FP (wasted offer):** one discount, e.g. **$50**.

So **one missed churner costs about 5× more than one wasted offer**. Trading one FN for a few FPs is still a profit, which is why the business leans towards **catching more churners (recall)**, even if a few more loyal customers get a discount (lower precision).


### Notes 3: ROC-AUC and PR-AUC (the two metrics we rank models on)

Both start from the model's **churn probability** (`predict_proba`), not from a yes/no prediction. Picture the model sorting all customers into a **risk list**, highest risk at the top. Every possible threshold is a cut-off point in that list: everyone above it gets an offer. Both curves show what happens at **every** cut-off, so they judge the **ranking**, not one threshold.

---

#### 1. Full forms and definitions

| | **ROC-AUC** | **PR-AUC** (sklearn: `average_precision`) |
|---|---|---|
| Full form | **R**eceiver **O**perating **C**haracteristic, **A**rea **U**nder the **C**urve | **P**recision–**R**ecall, **A**rea **U**nder the **C**urve |
| Plots | **TPR** (y) vs **FPR** (x) | **Precision** (y) vs **Recall** (x) |
| Formulas | TPR = TP / (TP + FN) = **recall**<br>FPR = FP / (FP + TN) | Precision = TP / (TP + FP)<br>Recall = TP / (TP + FN) |
| Definition | Probability that a **random churner** gets a **higher** risk score than a **random loyal customer** | Average precision over all recall levels: how "clean" the call list stays as we go further down it |
| Random model | **0.50** | **0.265** (= the churn rate) |
| Perfect model | 1.0 | 1.0 |
| Our LR (CV) | **0.85 ± 0.02** | **0.66 ± 0.05** |

**In Telco business words**
- **TPR (recall):** share of churners we reach with an offer. *Revenue saved.*
- **FPR:** share of **loyal** customers who get an offer they didn't need. *Wasted discounts.*
- **Precision:** share of the **call list** that really churns. *How many of the team's calls are worth making.*
- **ROC-AUC 0.85:** pick one customer who churned and one who stayed; 85% of the time the model puts the churner higher on the risk list.
- **PR-AUC 0.66:** as the retention team works down the list, on average about 2 of every 3 calls reach a real churner (random calling: about 1 in 4).

---

#### 2. What the charts look like

**ROC curve**
```
TPR   1.0 |        ____-----‾‾‾‾
(recall)  |    _-‾‾          .·
          |  /'           .·      ← diagonal = random guessing (AUC 0.5)
          | /   ← ours  .·
          |/  (AUC .85).·
      0.0 +·------------------
          0.0      FPR       1.0
```
- Each point on the curve is **one threshold**. Bottom-left: threshold 1.0, nobody flagged. Top-right: threshold 0, everyone flagged.
- **Closer to the top-left corner = better:** we catch many churners (high TPR) while bothering few loyal customers (low FPR).
- Our 0.5 cut-off is one point: about (FPR 0.09, TPR 0.53) in a fold. 70 of ≈ 776 loyal customers flagged, 148 of ≈ 280 churners caught.

**PR curve**
```
Precision 1.0 |‾‾\_
              |    ‾\__          ← ours (AUC .66)
              |        ‾‾\__
              |             ‾\_
        0.265 |- - - - - - - - -‾‾  ← random = churn rate (flat line)
              +-------------------
              0.0     Recall     1.0
```
- Read it **left to right as the call list getting longer**. On the left, only the top-risk customers are called and precision is high. Calling more customers raises recall, but more of the calls are wasted, so precision falls.
- **Higher and flatter = better:** the list stays "clean" for longer.
- At recall 1.0 (call everyone) precision drops to 0.265, because then the list is just the whole customer base.

**Why look at both?** Churners are the minority (26.5%). On the ROC chart, 70 wasted offers look small (FPR = 70 / 776 ≈ 9%), because the denominator is all loyal customers. On the PR chart, the same 70 are 70 of the 218 calls (≈ 32% wasted), which is what the retention team actually feels. **ROC-AUC = overall ranking skill; PR-AUC = quality of the call list.**

---

#### 3. Using them to compare models in CV and pick the best

**a) The numbers (what the comparison table uses)**
1. For each model, run `cross_validate` and take `test_roc_auc` and `test_pr_auc` as **mean ± std** over the 5 folds.
2. **Rank by mean ROC-AUC**; use **PR-AUC as the tie-breaker** (the spec's rule).
3. **Gap smaller than about one std → treat it as a tie.** Prefer the simpler, more explainable model (e.g. LR over XGBoost). If LR is within ~0.01 of the best boosted model, say so openly.
4. **Check overfitting:** with `return_train_score=True`, train ROC-AUC − CV ROC-AUC should be < ~0.05 (leakage check L6).
5. **Sanity check:** CV ROC-AUC > 0.90 on this dataset → suspect leakage, not a great model.

**b) The curves (the picture behind the numbers)**
- To draw CV curves without touching `X_test`, get **out-of-fold probabilities** on the training set: `cross_val_predict(pipe, X_train, y_train, cv=skf, method="predict_proba")[:, 1]`. Each customer is scored by a model that never saw them.
- Plot every model's ROC curve on one chart and every PR curve on another (`RocCurveDisplay.from_predictions`, `PrecisionRecallDisplay.from_predictions`).
- **One curve above another everywhere** → that model is better at every threshold. Easy choice.
- **Curves cross** → look at the region the business will use. If the team can only call the top ~20% of customers, compare the curves at **low FPR / low recall** (the left side), not the whole area.

**c) Business link**
- AUC picks the model that **ranks** churners best. That's the right job for phase 5, because the threshold isn't chosen yet.
- In **phase 6**, the best model's curve is where we pick **one point** (the threshold) by cost: $270 per missed churner vs $50 per wasted offer, within the team's capacity.
